# Carried Weight Parsing

## Bounded question

> How is carried weight represented in the source, and which values can be parsed reproducibly?

## Source and grain

This notebook examines the `wgt` field in the read-only SQLite source:

- **Database:** `data/raw/form_2015-present/form_2015-present/raceform.db`
- **Table:** `data`
- **Runner-record grain:** one physical source row per runner record
- **Physical source lineage:** preserve the original SQLite `rowid`
- **Data-row predicate:** `rowid <> 1`

The source contains 1,851,285 data-like runner records after excluding the imported header row at `rowid = 1`.

## Evidence-first method

The analysis will proceed from observed source values rather than assumed racing conventions. It will:

1. profile SQLite storage classes, missing values, sentinels and distinct raw values;
2. identify notation families and recurring structural patterns;
3. test candidate parsing rules against every observed value;
4. examine anomalies and ambiguous values in their runner, race and jurisdiction context;
5. distinguish exact source evidence from derived quantities;
6. preserve values that cannot be interpreted safely as unresolved rather than guessing;
7. define failure-detecting validation for both current and unseen future values.

Observations, interpretations and design consequences will be recorded separately.

## Explicit exclusions

This notebook will not:

- reopen race-distance analysis;
- alter or overwrite the raw `wgt` field;
- assume that every value uses British stones-and-pounds notation;
- infer undocumented allowances or corrections from weight alone;
- treat derived kilograms as independently verified official metric weights;
- design the final staging or target database schema;
- silently accept unfamiliar future notation.

## Expected database consequence

The expected result is a defensible parsing rule that allows downstream work to preserve separately:

- exact raw `wgt`;
- detected notation family;
- parsed stones;
- parsed pounds;
- candidate total pounds;
- candidate kilograms;
- parse status;
- ambiguity or anomaly flags.

Any derived total pounds or kilograms must retain explicit provenance as conversions of the source expression, not independently verified official values.

## Stopping rule

Stop when the notebook has established:

1. a reproducible rule covering every deterministically parseable current value;
2. documented known exceptions, ambiguities and malformed values;
3. a preservation mechanism for unresolved and future unseen notation;
4. failure-detecting validation that prevents unsupported values from being silently parsed;
5. a clear statement of which derived weight quantities are justified and their limitations.

In [1]:
# Import the standard libraries needed for the initial source-field profile.
import sqlite3
from pathlib import Path

import pandas as pd

# Resolve the project root from the notebook location.
PROJECT_ROOT = Path.cwd().resolve().parent
DB_PATH = PROJECT_ROOT / "data/raw/form_2015-present/form_2015-present/raceform.db"

DATA_ROW_PREDICATE = "rowid <> 1"

# Open the immutable source database explicitly in read-only mode.
connection = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)

# Profile the physical SQLite storage classes and basic missing-value states
# without making any assumptions about the meaning or notation of wgt.
wgt_storage_profile = pd.read_sql_query(
    f"""
    SELECT
        typeof(wgt) AS sqlite_storage_class,
        COUNT(*) AS runner_records,
        SUM(CASE WHEN wgt IS NULL THEN 1 ELSE 0 END) AS sql_null_rows,
        SUM(CASE WHEN trim(CAST(wgt AS TEXT)) = '' THEN 1 ELSE 0 END) AS blank_text_rows
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY typeof(wgt)
    ORDER BY runner_records DESC
    """,
    connection,
)

wgt_storage_profile

,sqlite_storage_class,runner_records,sql_null_rows,blank_text_rows
0,text,1851285,0,0


In [2]:
# Summarise the distinct raw wgt values exactly as stored.
# This reveals the dominant formats, rare values, and possible sentinels
# without normalising or interpreting the source text.
wgt_value_profile = pd.read_sql_query(
    f"""
    SELECT
        wgt AS raw_wgt,
        COUNT(*) AS runner_records,
        COUNT(DISTINCT date || '|' || course || '|' || off) AS provisional_races
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY wgt
    ORDER BY runner_records DESC, raw_wgt
    """,
    connection,
)

print(f"Distinct raw wgt values: {len(wgt_value_profile):,}")
wgt_value_profile

Distinct raw wgt values: 79


,raw_wgt,runner_records,provisional_races
0,9-0,127748,49068
1,9-2,117269,50873
2,9-5,81374,41224
3,8-11,74459,38745
4,9-7,71569,43605
...,...,...,...
74,12-9,3,3
75,12-10,1,1
76,12-11,1,1
77,6-12,1,1


In [3]:
# Classify every distinct raw value by its exact textual structure.
# This does not yet interpret either side of the hyphen as a unit.
import re

def classify_wgt_structure(raw_wgt: str) -> str:
    """Return a structural notation family without assigning racing meaning."""
    if re.fullmatch(r"\d+-\d+", raw_wgt):
        return "integer-hyphen-integer"
    if re.fullmatch(r"\d+", raw_wgt):
        return "integer-only"
    if re.fullmatch(r"\d+(?:\.\d+)?", raw_wgt):
        return "decimal-only"
    if re.fullmatch(r"\d+\s*[a-zA-Z]+", raw_wgt):
        return "number-with-unit-text"
    return "other-or-ambiguous"

wgt_value_profile["notation_structure"] = (
    wgt_value_profile["raw_wgt"]
    .astype(str)
    .map(classify_wgt_structure)
)

wgt_structure_summary = (
    wgt_value_profile
    .groupby("notation_structure", as_index=False)
    .agg(
        distinct_raw_values=("raw_wgt", "nunique"),
        runner_records=("runner_records", "sum"),
        provisional_races=("provisional_races", "sum"),
    )
    .sort_values(
        ["runner_records", "distinct_raw_values"],
        ascending=[False, False],
    )
)

print("Structural summary:")
display(wgt_structure_summary)

print("\nAll distinct raw values:")
display(
    wgt_value_profile[
        ["raw_wgt", "runner_records", "provisional_races", "notation_structure"]
    ].sort_values(
        ["notation_structure", "raw_wgt"],
        kind="stable",
    )
)

Structural summary:


,notation_structure,distinct_raw_values,runner_records,provisional_races
0,integer-hyphen-integer,79,1851285,1120232



All distinct raw values:


,raw_wgt,runner_records,provisional_races,notation_structure
38,10-0,18144,14737,integer-hyphen-integer
51,10-1,10544,8399,integer-hyphen-integer
28,10-10,21212,13876,integer-hyphen-integer
25,10-11,21455,15093,integer-hyphen-integer
17,10-12,35570,18031,integer-hyphen-integer
...,...,...,...,...
2,9-5,81374,41224,integer-hyphen-integer
8,9-6,58025,37081,integer-hyphen-integer
4,9-7,71569,43605,integer-hyphen-integer
20,9-8,33285,23042,integer-hyphen-integer


In [4]:
# Split every distinct observed value into its two integer components.
# We are testing whether the structure is compatible with stones and pounds,
# but we are not yet declaring that interpretation proven.

wgt_components = wgt_value_profile.copy()

wgt_components[["left_component", "right_component"]] = (
    wgt_components["raw_wgt"]
    .str.extract(r"^(\d+)-(\d+)$")
    .astype(int)
)

component_summary = pd.DataFrame(
    {
        "measure": [
            "minimum left component",
            "maximum left component",
            "minimum right component",
            "maximum right component",
            "distinct left components",
            "distinct right components",
            "values with right component above 13",
            "runner records with right component above 13",
        ],
        "value": [
            wgt_components["left_component"].min(),
            wgt_components["left_component"].max(),
            wgt_components["right_component"].min(),
            wgt_components["right_component"].max(),
            wgt_components["left_component"].nunique(),
            wgt_components["right_component"].nunique(),
            int((wgt_components["right_component"] > 13).sum()),
            int(
                wgt_components.loc[
                    wgt_components["right_component"] > 13,
                    "runner_records",
                ].sum()
            ),
        ],
    }
)

display(component_summary)

print("Observed right components:")
display(
    wgt_components.groupby("right_component", as_index=False).agg(
        distinct_raw_values=("raw_wgt", "nunique"),
        runner_records=("runner_records", "sum"),
    )
)

print("\nObserved left components:")
display(
    wgt_components.groupby("left_component", as_index=False).agg(
        distinct_raw_values=("raw_wgt", "nunique"),
        runner_records=("runner_records", "sum"),
    )
)

,measure,value
0,minimum left component,6
1,maximum left component,12
2,minimum right component,0
3,maximum right component,13
4,distinct left components,7
5,distinct right components,14
6,values with right component above 13,0
7,runner records with right component above 13,0


Observed right components:


,right_component,distinct_raw_values,runner_records
0,0,5,207304
1,1,5,91883
2,2,5,174413
3,3,5,108398
4,4,5,130547
5,5,6,144082
6,6,6,126493
7,7,6,162662
8,8,6,110193
9,9,6,133315



Observed left components:


,left_component,distinct_raw_values,runner_records
0,6,2,2
1,7,9,7624
2,8,14,452589
3,9,14,789098
4,10,14,252569
5,11,14,332457
6,12,12,16946


In [5]:
# Calculate candidate total pounds for each distinct raw value under the
# stones-and-pounds interpretation, then inspect the lightest and heaviest
# observed values in their source context.
wgt_components["candidate_total_pounds"] = (
    wgt_components["left_component"] * 14
    + wgt_components["right_component"]
)

candidate_range_summary = pd.DataFrame(
    {
        "measure": [
            "minimum candidate total pounds",
            "maximum candidate total pounds",
            "minimum raw wgt",
            "maximum raw wgt",
            "distinct candidate total-pound values",
        ],
        "value": [
            wgt_components["candidate_total_pounds"].min(),
            wgt_components["candidate_total_pounds"].max(),
            wgt_components.loc[
                wgt_components["candidate_total_pounds"].idxmin(),
                "raw_wgt",
            ],
            wgt_components.loc[
                wgt_components["candidate_total_pounds"].idxmax(),
                "raw_wgt",
            ],
            wgt_components["candidate_total_pounds"].nunique(),
        ],
    }
)

display(candidate_range_summary)

# Select the five lightest and five heaviest distinct observed weights.
extreme_weights = (
    pd.concat(
        [
            wgt_components.nsmallest(5, "candidate_total_pounds"),
            wgt_components.nlargest(5, "candidate_total_pounds"),
        ]
    )
    ["raw_wgt"]
    .drop_duplicates()
    .tolist()
)

placeholders = ", ".join("?" for _ in extreme_weights)

extreme_weight_context = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        race_name,
        type,
        horse,
        wgt AS raw_wgt
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND wgt IN ({placeholders})
    ORDER BY
        CAST(substr(wgt, 1, instr(wgt, '-') - 1) AS INTEGER) * 14
            + CAST(substr(wgt, instr(wgt, '-') + 1) AS INTEGER),
        date,
        course,
        off,
        rowid
    """,
    connection,
    params=extreme_weights,
)

print("Runner-record counts for the extreme observed weights:")
display(
    extreme_weight_context.groupby(
        ["raw_wgt", "type"],
        as_index=False,
        dropna=False,
    ).agg(
        runner_records=("source_rowid", "size"),
        provisional_races=("source_rowid", "count"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
)

print("\nSource examples for the extreme observed weights:")
display(extreme_weight_context)

,measure,value
0,minimum candidate total pounds,96
1,maximum candidate total pounds,179
2,minimum raw wgt,6-12
3,maximum raw wgt,12-11
4,distinct candidate total-pound values,79


Runner-record counts for the extreme observed weights:


,raw_wgt,type,runner_records,provisional_races,first_date,last_date
0,12-10,Chase,1,1,2024-12-28,2024-12-28
1,12-11,Chase,1,1,2016-10-04,2016-10-04
2,12-7,Chase,54,54,2015-05-01,2025-10-12
3,12-7,Hurdle,19,19,2018-02-10,2026-02-26
4,12-8,Chase,16,16,2015-02-06,2026-05-13
5,12-9,Chase,2,2,2021-04-30,2024-05-31
6,12-9,Hurdle,1,1,2021-05-18,2021-05-18
7,6-12,Flat,1,1,2019-09-17,2019-09-17
8,6-13,Flat,1,1,2019-02-16,2019-02-16
9,7-5,Flat,9,9,2017-01-30,2017-06-11



Source examples for the extreme observed weights:


,source_rowid,date,course,off,race_name,type,horse,raw_wgt
0,777412,2019-09-17,Galway (IRE),5:15,James P. Cunningham Electrical Handicap,Flat,Hattie Amarin (IRE),6-12
1,670715,2019-02-16,Flemington (AUS),5:50,Black Caviar Lightning (2yo+) (Turf),Flat,Jedastar (AUS),6-13
2,325961,2017-01-30,Sha Tin (HK),8:05,Chinese New Year Cup (Handicap) (3yo+) (Cours...,Flat,Flying Moochi (USA),7-5
3,336736,2017-03-01,Sha Tin (HK),2:20,Flamingo Handicap (3yo+) (All Weather Track) ...,Flat,Startling Power (AUS),7-5
4,347766,2017-03-29,Happy Valley (HK),12:45,Matheson Handicap (3yo+) (Course C+3) (Turf),Flat,King Of Smarts (AUS),7-5
...,...,...,...,...,...,...,...,...
228,1002048,2021-04-30,Punchestown (IRE),3:40,Paddy Power Hunters Chase for the Bishopscourt...,Chase,Rewritetherules (IRE),12-9
229,1013366,2021-05-18,Hexham,7:05,Hexham Racecourse Static Caravan Sites Availab...,Hurdle,Valse Au Taillons (FR),12-9
230,1515055,2024-05-31,Stratford,8:05,Peter Wright Over And Out Handicap Hunters Chase,Chase,Stratagem (FR),12-9
231,1617323,2024-12-28,Catterick,12:25,Sky Bet Build A Bet Conditional Jockeys Handic...,Chase,Not Staying Long (IRE),12-10


In [6]:
# Profile candidate carried weight by the source race-type field.
# This tests whether the stones-and-pounds interpretation produces
# contextually coherent ranges without yet declaring every record valid.

wgt_by_type = pd.read_sql_query(
    f"""
    WITH parsed AS (
        SELECT
            type AS race_type,
            wgt AS raw_wgt,
            CAST(substr(wgt, 1, instr(wgt, '-') - 1) AS INTEGER) AS stones,
            CAST(substr(wgt, instr(wgt, '-') + 1) AS INTEGER) AS pounds
        FROM data
        WHERE {DATA_ROW_PREDICATE}
    ),
    derived AS (
        SELECT
            race_type,
            raw_wgt,
            stones,
            pounds,
            (stones * 14) + pounds AS candidate_total_pounds
        FROM parsed
    )
    SELECT
        race_type,
        COUNT(*) AS runner_records,
        COUNT(DISTINCT raw_wgt) AS distinct_raw_values,
        MIN(candidate_total_pounds) AS minimum_total_pounds,
        MAX(candidate_total_pounds) AS maximum_total_pounds,
        MIN(stones) AS minimum_stones,
        MAX(stones) AS maximum_stones,
        MIN(pounds) AS minimum_pounds_component,
        MAX(pounds) AS maximum_pounds_component
    FROM derived
    GROUP BY race_type
    ORDER BY runner_records DESC, race_type
    """,
    connection,
)

display(wgt_by_type)

# Show the exact lightest and heaviest values within each race type,
# including their frequencies, so isolated boundary records remain visible.
wgt_type_boundaries = pd.read_sql_query(
    f"""
    WITH parsed AS (
        SELECT
            type AS race_type,
            wgt AS raw_wgt,
            (CAST(substr(wgt, 1, instr(wgt, '-') - 1) AS INTEGER) * 14)
              + CAST(substr(wgt, instr(wgt, '-') + 1) AS INTEGER)
                AS candidate_total_pounds
        FROM data
        WHERE {DATA_ROW_PREDICATE}
    ),
    counts AS (
        SELECT
            race_type,
            raw_wgt,
            candidate_total_pounds,
            COUNT(*) AS runner_records
        FROM parsed
        GROUP BY race_type, raw_wgt, candidate_total_pounds
    ),
    limits AS (
        SELECT
            race_type,
            MIN(candidate_total_pounds) AS minimum_total_pounds,
            MAX(candidate_total_pounds) AS maximum_total_pounds
        FROM counts
        GROUP BY race_type
    )
    SELECT
        c.race_type,
        c.raw_wgt,
        c.candidate_total_pounds,
        c.runner_records,
        CASE
            WHEN c.candidate_total_pounds = l.minimum_total_pounds
             AND c.candidate_total_pounds = l.maximum_total_pounds
                THEN 'minimum and maximum'
            WHEN c.candidate_total_pounds = l.minimum_total_pounds
                THEN 'minimum'
            WHEN c.candidate_total_pounds = l.maximum_total_pounds
                THEN 'maximum'
        END AS boundary
    FROM counts AS c
    JOIN limits AS l
      ON c.race_type = l.race_type
    WHERE c.candidate_total_pounds IN (
        l.minimum_total_pounds,
        l.maximum_total_pounds
    )
    ORDER BY c.race_type, c.candidate_total_pounds, c.raw_wgt
    """,
    connection,
)

display(wgt_type_boundaries)

,race_type,runner_records,distinct_raw_values,minimum_total_pounds,maximum_total_pounds,minimum_stones,maximum_stones,minimum_pounds_component,maximum_pounds_component
0,Flat,1268229,73,96,174,6,12,0,13
1,Hurdle,358441,51,125,177,8,12,0,13
2,Chase,179645,58,114,179,8,12,0,13
3,NH Flat,44970,42,131,172,9,12,0,13


,race_type,raw_wgt,candidate_total_pounds,runner_records,boundary
0,Chase,8-2,114,1,minimum
1,Chase,12-11,179,1,maximum
2,Flat,6-12,96,1,minimum
3,Flat,12-6,174,1,maximum
4,Hurdle,8-13,125,2,minimum
5,Hurdle,12-9,177,1,maximum
6,NH Flat,9-5,131,1,minimum
7,NH Flat,12-4,172,2,maximum


In [7]:
# Retrieve every runner record at the minimum or maximum candidate weight
# within its source race type. This keeps the next judgement grounded in
# the full race and runner context rather than the aggregate range alone.

race_type_boundary_records = pd.read_sql_query(
    f"""
    WITH parsed AS (
        SELECT
            rowid AS source_rowid,
            date,
            course,
            off,
            race_id,
            race_name,
            type AS race_type,
            ran,
            num,
            pos,
            horse,
            jockey,
            trainer,
            wgt AS raw_wgt,
            (CAST(substr(wgt, 1, instr(wgt, '-') - 1) AS INTEGER) * 14)
              + CAST(substr(wgt, instr(wgt, '-') + 1) AS INTEGER)
                AS candidate_total_pounds
        FROM data
        WHERE {DATA_ROW_PREDICATE}
    ),
    limits AS (
        SELECT
            race_type,
            MIN(candidate_total_pounds) AS minimum_total_pounds,
            MAX(candidate_total_pounds) AS maximum_total_pounds
        FROM parsed
        GROUP BY race_type
    )
    SELECT
        p.*,
        CASE
            WHEN p.candidate_total_pounds = l.minimum_total_pounds
             AND p.candidate_total_pounds = l.maximum_total_pounds
                THEN 'minimum and maximum'
            WHEN p.candidate_total_pounds = l.minimum_total_pounds
                THEN 'minimum'
            WHEN p.candidate_total_pounds = l.maximum_total_pounds
                THEN 'maximum'
        END AS boundary
    FROM parsed AS p
    JOIN limits AS l
      ON p.race_type = l.race_type
    WHERE p.candidate_total_pounds IN (
        l.minimum_total_pounds,
        l.maximum_total_pounds
    )
    ORDER BY
        p.race_type,
        p.candidate_total_pounds,
        p.date,
        p.course,
        p.off,
        p.source_rowid
    """,
    connection,
)

race_type_boundary_records

,source_rowid,date,course,off,race_id,race_name,race_type,ran,num,pos,horse,jockey,trainer,raw_wgt,candidate_total_pounds,boundary
0,412964,2017-08-04,Bath,7:40,680528,Kingstone Press Wild Berry Chase Handicap (Bat...,Chase,10,13,7,Rolanna (IRE),Royston Ffrench,W J Martin,8-2,114,minimum
1,276165,2016-10-04,Tipperary (IRE),4:55,660282,Tipperary Handicap Chase,Chase,9,1,5,Usa (IRE),Donagh Meyler,S J Mahon,12-11,179,maximum
2,777412,2019-09-17,Galway (IRE),5:15,740274,James P. Cunningham Electrical Handicap,Flat,17,19,13,Hattie Amarin (IRE),Mikey Sheehy,W McCreery,6-12,96,minimum
3,424396,2017-08-28,Ostend (BEL),5:55,683522,Prijs S. Kervyn dOud Mooreghem (Handicap) (4yo...,Flat,9,1,5,Alcatraz (IRE),Mme Melissa Boisgontier,George Baker,12-6,174,maximum
4,215743,2016-05-30,Les Landes (JER),2:30,652781,The Presidents Hurdle (Handicap),Hurdle,5,0,1,Cahill (IRE),Alice Mills,Mrs C Gilbert,8-13,125,minimum
5,440380,2017-09-30,Gowran Park (IRE),2:15,685384,Thomastown Handicap Hurdle,Hurdle,13,13,12,Coral Pearl (IRE),Cathal Landers,S M Duffy,8-13,125,minimum
6,1013366,2021-05-18,Hexham,7:05,782558,Hexham Racecourse Static Caravan Sites Availab...,Hurdle,5,1,1,Valse Au Taillons (FR),Kevin Brogan,Johnny Farrelly,12-9,177,maximum
7,520192,2018-04-14,Chepstow,6:05,696797,Collect totepool Winnings At Betfred Shops Sta...,NH Flat,9,9,2,Nikap (FR),Tom Buckley,Nigel Hawke,9-5,131,minimum
8,1014354,2021-05-20,Tipperary (IRE),5:15,785281,Racing Again June 1st (Pro/Am) INH Flat Race,NH Flat,13,2,2,Kottayam (IRE),Mr P W Mullins,W P Mullins,12-4,172,maximum
9,1014450,2021-05-20,Tipperary (IRE),5:15,785281,Racing Again June 1st (Pro/Am) INH Flat Race,NH Flat,13,1,4,John Cannon (GB),Mr J J Codd,Joseph G Murphy,12-4,172,maximum


In [8]:
# Retrieve every runner from each race containing a race-type boundary record.
# This tests whether the unusual weight is coherent within its race and helps
# separate valid weight notation from possible anomalies in other source fields.

boundary_race_keys = (
    race_type_boundary_records[
        ["date", "course", "off"]
    ]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)

boundary_race_keys = list(boundary_race_keys)

race_conditions = " OR ".join(
    "(date = ? AND course = ? AND off = ?)"
    for _ in boundary_race_keys
)

race_params = [
    value
    for race_key in boundary_race_keys
    for value in race_key
]

boundary_race_context = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        race_name,
        type AS race_type,
        ran,
        num,
        pos,
        horse,
        jockey,
        wgt AS raw_wgt,
        (CAST(substr(wgt, 1, instr(wgt, '-') - 1) AS INTEGER) * 14)
          + CAST(substr(wgt, instr(wgt, '-') + 1) AS INTEGER)
            AS candidate_total_pounds
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND ({race_conditions})
    ORDER BY
        date,
        course,
        off,
        candidate_total_pounds DESC,
        num,
        source_rowid
    """,
    connection,
    params=race_params,
)

display(
    boundary_race_context.groupby(
        ["date", "course", "off", "race_name", "race_type"],
        as_index=False,
    ).agg(
        runner_records=("source_rowid", "size"),
        stated_ran=("ran", "first"),
        distinct_weights=("raw_wgt", "nunique"),
        minimum_total_pounds=("candidate_total_pounds", "min"),
        maximum_total_pounds=("candidate_total_pounds", "max"),
    )
)

display(boundary_race_context)

,date,course,off,race_name,race_type,runner_records,stated_ran,distinct_weights,minimum_total_pounds,maximum_total_pounds
0,2016-05-30,Les Landes (JER),2:30,The Presidents Hurdle (Handicap),Hurdle,5,5,4,125,164
1,2016-10-04,Tipperary (IRE),4:55,Tipperary Handicap Chase,Chase,9,9,7,134,179
2,2017-08-04,Bath,7:40,Kingstone Press Wild Berry Chase Handicap (Bat...,Chase,10,10,9,114,134
3,2017-08-28,Ostend (BEL),5:55,Prijs S. Kervyn dOud Mooreghem (Handicap) (4yo...,Flat,9,9,7,128,174
4,2017-09-30,Gowran Park (IRE),2:15,Thomastown Handicap Hurdle,Hurdle,13,13,11,125,161
5,2018-04-14,Chepstow,6:05,Collect totepool Winnings At Betfred Shops Sta...,NH Flat,9,9,6,131,154
6,2019-09-17,Galway (IRE),5:15,James P. Cunningham Electrical Handicap,Flat,17,17,15,96,139
7,2021-05-18,Hexham,7:05,Hexham Racecourse Static Caravan Sites Availab...,Hurdle,5,5,4,149,177
8,2021-05-20,Tipperary (IRE),5:15,Racing Again June 1st (Pro/Am) INH Flat Race,NH Flat,13,13,6,151,172


,source_rowid,date,course,off,race_name,race_type,ran,num,pos,horse,jockey,raw_wgt,candidate_total_pounds
0,215745,2016-05-30,Les Landes (JER),2:30,The Presidents Hurdle (Handicap),Hurdle,5,0,4,Drum Valley (GB),Mattie Batchelor,11-10,164
1,215746,2016-05-30,Les Landes (JER),2:30,The Presidents Hurdle (Handicap),Hurdle,5,0,5,Granit Man (FR),Richard Condon,9-6,132
2,215744,2016-05-30,Les Landes (JER),2:30,The Presidents Hurdle (Handicap),Hurdle,5,0,3,Fourni (IRE),Miss M Hooper,9-3,129
3,215750,2016-05-30,Les Landes (JER),2:30,The Presidents Hurdle (Handicap),Hurdle,5,0,2,Dalmo (GB),Philip Prince,9-3,129
4,215743,2016-05-30,Les Landes (JER),2:30,The Presidents Hurdle (Handicap),Hurdle,5,0,1,Cahill (IRE),Alice Mills,8-13,125
...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,1014510,2021-05-20,Tipperary (IRE),5:15,Racing Again June 1st (Pro/Am) INH Flat Race,NH Flat,13,9,8,Aghinagh (IRE),Mr Pat Taaffe,10-12,152
86,1014503,2021-05-20,Tipperary (IRE),5:15,Racing Again June 1st (Pro/Am) INH Flat Race,NH Flat,13,10,10,Benny The Duke (IRE),Jack Foley,10-12,152
87,1014501,2021-05-20,Tipperary (IRE),5:15,Racing Again June 1st (Pro/Am) INH Flat Race,NH Flat,13,18,12,Brownstown Head (IRE),Mr D Kiely,10-12,152
88,1014416,2021-05-20,Tipperary (IRE),5:15,Racing Again June 1st (Pro/Am) INH Flat Race,NH Flat,13,15,3,Bang On Target (IRE),T Joseph Kelly,10-11,151


In [9]:
# Calculate the candidate weight range within every provisional race.
# Large within-race spreads are not automatically errors, but they provide
# a systematic way to find unusual contexts that may require anomaly flags.

race_weight_spreads = pd.read_sql_query(
    f"""
    WITH parsed AS (
        SELECT
            rowid AS source_rowid,
            date,
            course,
            off,
            race_name,
            type AS race_type,
            ran,
            wgt AS raw_wgt,
            (CAST(substr(wgt, 1, instr(wgt, '-') - 1) AS INTEGER) * 14)
              + CAST(substr(wgt, instr(wgt, '-') + 1) AS INTEGER)
                AS candidate_total_pounds
        FROM data
        WHERE {DATA_ROW_PREDICATE}
    ),
    race_summary AS (
        SELECT
            date,
            course,
            off,
            MIN(race_name) AS race_name,
            MIN(race_type) AS race_type,
            COUNT(*) AS runner_records,
            MIN(ran) AS minimum_stated_ran,
            MAX(ran) AS maximum_stated_ran,
            COUNT(DISTINCT raw_wgt) AS distinct_weights,
            MIN(candidate_total_pounds) AS minimum_total_pounds,
            MAX(candidate_total_pounds) AS maximum_total_pounds,
            MAX(candidate_total_pounds) - MIN(candidate_total_pounds)
                AS weight_spread_pounds
        FROM parsed
        GROUP BY date, course, off
    )
    SELECT *
    FROM race_summary
    ORDER BY
        weight_spread_pounds DESC,
        date,
        course,
        off
    """,
    connection,
)

spread_summary = pd.DataFrame(
    {
        "measure": [
            "provisional races",
            "minimum race weight spread",
            "median race weight spread",
            "95th percentile race weight spread",
            "99th percentile race weight spread",
            "maximum race weight spread",
            "races with zero spread",
            "races with spread above 42 lb",
            "races with inconsistent stated ran",
            "races where runner records differ from stated ran",
        ],
        "value": [
            len(race_weight_spreads),
            race_weight_spreads["weight_spread_pounds"].min(),
            race_weight_spreads["weight_spread_pounds"].median(),
            race_weight_spreads["weight_spread_pounds"].quantile(0.95),
            race_weight_spreads["weight_spread_pounds"].quantile(0.99),
            race_weight_spreads["weight_spread_pounds"].max(),
            int((race_weight_spreads["weight_spread_pounds"] == 0).sum()),
            int((race_weight_spreads["weight_spread_pounds"] > 42).sum()),
            int(
                (
                    race_weight_spreads["minimum_stated_ran"]
                    != race_weight_spreads["maximum_stated_ran"]
                ).sum()
            ),
            int(
                (
                    race_weight_spreads["runner_records"]
                    != race_weight_spreads["maximum_stated_ran"]
                ).sum()
            ),
        ],
    }
)

display(spread_summary)

print("Races with the largest observed carried-weight spreads:")
display(race_weight_spreads.head(25))

,measure,value
0,provisional races,189043.0
1,minimum race weight spread,0.0
2,median race weight spread,14.0
3,95th percentile race weight spread,28.0
4,99th percentile race weight spread,33.0
5,maximum race weight spread,46.0
6,races with zero spread,9967.0
7,races with spread above 42 lb,4.0
8,races with inconsistent stated ran,0.0
9,races where runner records differ from stated ran,5.0


Races with the largest observed carried-weight spreads:


,date,course,off,race_name,race_type,runner_records,minimum_stated_ran,maximum_stated_ran,distinct_weights,minimum_total_pounds,maximum_total_pounds,weight_spread_pounds
0,2017-08-28,Ostend (BEL),5:55,Prijs S. Kervyn dOud Mooreghem (Handicap) (4yo...,Flat,9,9,9,7,128,174,46
1,2016-10-04,Tipperary (IRE),4:55,Tipperary Handicap Chase,Chase,9,9,9,7,134,179,45
2,2016-05-01,Sligo (IRE),3:30,Durkin Bros. Electrical Gurteen Handicap,Flat,14,14,14,12,106,149,43
3,2019-09-17,Galway (IRE),5:15,James P. Cunningham Electrical Handicap,Flat,17,17,17,15,96,139,43
4,2015-04-06,Les Landes (JER),3:40,Jersey Bookmakers Handicap,Flat,11,11,11,8,117,159,42
5,2019-05-14,Ffos Las,5:30,Tim Vaughan Racing Supporting Ffos Las Mares H...,Hurdle,8,8,8,8,133,175,42
6,2019-07-30,Perth,9:00,perthlodge.co.uk Amateur Riders Handicap Hurdle,Hurdle,10,10,10,9,133,175,42
7,2022-04-08,Sedgefield,5:08,bet365 Handicap Chase,Chase,10,10,10,7,133,175,42
8,2025-05-24,Les Landes (JER),3:05,Millbrook Sprint (Handicap) (Turf),Flat,5,5,5,5,110,152,42
9,2017-08-04,Bangor-on-Dee,3:10,Fraser Wealth Management Handicap Chase,Chase,11,11,11,11,133,174,41


In [10]:
# Inspect every runner in the races whose carried-weight spread exceeds 42 lb.
# These are the most extreme current contexts and therefore the strongest
# candidates for identifying source anomalies separate from parse validity.

widest_race_keys = list(
    race_weight_spreads.loc[
        race_weight_spreads["weight_spread_pounds"] > 42,
        ["date", "course", "off"],
    ].itertuples(index=False, name=None)
)

widest_race_conditions = " OR ".join(
    "(date = ? AND course = ? AND off = ?)"
    for _ in widest_race_keys
)

widest_race_params = [
    value
    for race_key in widest_race_keys
    for value in race_key
]

widest_race_context = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        race_name,
        type AS race_type,
        ran,
        num,
        pos,
        horse,
        jockey,
        trainer,
        wgt AS raw_wgt,
        (CAST(substr(wgt, 1, instr(wgt, '-') - 1) AS INTEGER) * 14)
          + CAST(substr(wgt, instr(wgt, '-') + 1) AS INTEGER)
            AS candidate_total_pounds
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND ({widest_race_conditions})
    ORDER BY
        date,
        course,
        off,
        candidate_total_pounds DESC,
        num,
        source_rowid
    """,
    connection,
    params=widest_race_params,
)

widest_race_context

,source_rowid,date,course,off,race_name,race_type,ran,num,pos,horse,jockey,trainer,raw_wgt,candidate_total_pounds
0,200354,2016-05-01,Sligo (IRE),3:30,Durkin Bros. Electrical Gurteen Handicap,Flat,14,1,9,Dont Bother Me (IRE),A E Lynch,Niall Moran,10-9,149
1,200328,2016-05-01,Sligo (IRE),3:30,Durkin Bros. Electrical Gurteen Handicap,Flat,14,3,1,Downforce (IRE),W J Lee,W McCreery,9-9,135
2,200325,2016-05-01,Sligo (IRE),3:30,Durkin Bros. Electrical Gurteen Handicap,Flat,14,4,4,Hidden Oasis (IRE),Wayne Lordan,David Wachman,9-7,133
3,200326,2016-05-01,Sligo (IRE),3:30,Durkin Bros. Electrical Gurteen Handicap,Flat,14,5,3,Bainne (IRE),Kieren Fallon,M D OCallaghan,9-6,132
4,200382,2016-05-01,Sligo (IRE),3:30,Durkin Bros. Electrical Gurteen Handicap,Flat,14,2,8,Roachdale House (IRE),Danny Redmond,Paul W Flynn,9-5,131
5,200379,2016-05-01,Sligo (IRE),3:30,Durkin Bros. Electrical Gurteen Handicap,Flat,14,6,13,Burren View Lady (IRE),Connor King,Denis Gerard Hogan,9-5,131
6,200324,2016-05-01,Sligo (IRE),3:30,Durkin Bros. Electrical Gurteen Handicap,Flat,14,7,5,Ducky Mallon (IRE),Gary Carroll,Donal Kinsella,9-3,129
7,200378,2016-05-01,Sligo (IRE),3:30,Durkin Bros. Electrical Gurteen Handicap,Flat,14,9,14,Caigemdar (IRE),Colin Keane,Mark Michael McNiff,8-13,125
8,200380,2016-05-01,Sligo (IRE),3:30,Durkin Bros. Electrical Gurteen Handicap,Flat,14,8,12,Lilys Prince (IRE),Robbie Downey,Garvan Donnelly,8-12,124
9,200353,2016-05-01,Sligo (IRE),3:30,Durkin Bros. Electrical Gurteen Handicap,Flat,14,11,7,Enter The Red (IRE),Leigh Roche,Aidan Anthony Howard,8-8,120


In [11]:
# Test whether the observed stones-and-pounds strings are canonical and
# map one-to-one onto candidate total pounds.
#
# A canonical value must:
# - contain no surrounding whitespace;
# - contain no leading zeros;
# - use exactly one hyphen;
# - use a pounds component from 0 through 13;
# - reconstruct exactly from its parsed components.
#
# We also check whether two different raw strings ever produce the same
# candidate total pounds.

canonical_weight_check = wgt_components.copy()

canonical_weight_check["reconstructed_raw_wgt"] = (
    canonical_weight_check["left_component"].astype(str)
    + "-"
    + canonical_weight_check["right_component"].astype(str)
)

canonical_weight_check["is_canonical"] = (
    canonical_weight_check["raw_wgt"]
    == canonical_weight_check["reconstructed_raw_wgt"]
)

total_pound_collisions = (
    canonical_weight_check
    .groupby("candidate_total_pounds", as_index=False)
    .agg(
        distinct_raw_values=("raw_wgt", "nunique"),
        raw_values=("raw_wgt", lambda values: ", ".join(sorted(values))),
        runner_records=("runner_records", "sum"),
    )
    .query("distinct_raw_values > 1")
)

observed_total_pounds = set(
    canonical_weight_check["candidate_total_pounds"]
)

complete_candidate_range = set(
    range(
        canonical_weight_check["candidate_total_pounds"].min(),
        canonical_weight_check["candidate_total_pounds"].max() + 1,
    )
)

missing_totals_inside_range = sorted(
    complete_candidate_range - observed_total_pounds
)

canonical_summary = pd.DataFrame(
    {
        "measure": [
            "distinct raw values",
            "canonical raw values",
            "non-canonical raw values",
            "distinct candidate total pounds",
            "total-pound collisions",
            "minimum candidate total pounds",
            "maximum candidate total pounds",
            "unobserved totals inside current range",
        ],
        "value": [
            len(canonical_weight_check),
            int(canonical_weight_check["is_canonical"].sum()),
            int((~canonical_weight_check["is_canonical"]).sum()),
            canonical_weight_check["candidate_total_pounds"].nunique(),
            len(total_pound_collisions),
            canonical_weight_check["candidate_total_pounds"].min(),
            canonical_weight_check["candidate_total_pounds"].max(),
            len(missing_totals_inside_range),
        ],
    }
)

display(canonical_summary)

print("Unobserved total-pound values inside the current observed range:")
display(
    pd.DataFrame(
        {
            "candidate_total_pounds": missing_totals_inside_range,
            "canonical_stones_pounds": [
                f"{total // 14}-{total % 14}"
                for total in missing_totals_inside_range
            ],
        }
    )
)

print("\nAny collisions where different raw values map to one total:")
display(total_pound_collisions)

,measure,value
0,distinct raw values,79
1,canonical raw values,79
2,non-canonical raw values,0
3,distinct candidate total pounds,79
4,total-pound collisions,0
5,minimum candidate total pounds,96
6,maximum candidate total pounds,179
7,unobserved totals inside current range,5


Unobserved total-pound values inside the current observed range:


,candidate_total_pounds,canonical_stones_pounds
0,98,7-0
1,99,7-1
2,100,7-2
3,101,7-3
4,102,7-4



Any collisions where different raw values map to one total:


,candidate_total_pounds,distinct_raw_values,raw_values,runner_records


In [12]:
# Profile raw carried-weight notation and candidate ranges by course jurisdiction.
# This tests whether jurisdictions that normally publish weights metrically
# have also been standardised into stones-and-pounds in this source.
#
# The course-to-jurisdiction mapping was established in Notebook 04, so first
# inspect the reusable project modules to identify the existing mapping helper
# rather than recreating jurisdiction logic in this notebook.

from pathlib import Path

source_module_files = sorted(
    (PROJECT_ROOT / "src" / "inside_rails").glob("*.py")
)

pd.DataFrame(
    {
        "module_file": [
            path.relative_to(PROJECT_ROOT).as_posix()
            for path in source_module_files
        ]
    }
)

,module_file
0,src/inside_rails/__init__.py
1,src/inside_rails/race_distance.py
2,src/inside_rails/source_sqlite.py


In [13]:
# Profile carried-weight representation across broad jurisdiction groups inferred
# directly from the source course text.
#
# This is a source-text classification for analysis only. It does not replace
# the fuller course-jurisdiction work completed elsewhere, and unmatched courses
# remain explicit rather than being guessed.

jurisdiction_weight_profile = pd.read_sql_query(
    f"""
    WITH classified AS (
        SELECT
            rowid AS source_rowid,
            date,
            course,
            off,
            wgt AS raw_wgt,
            CASE
                WHEN course LIKE '% (IRE)' THEN 'Ireland'
                WHEN course LIKE '% (AUS)' THEN 'Australia'
                WHEN course LIKE '% (HK)' THEN 'Hong Kong'
                WHEN course LIKE '% (FR)' THEN 'France'
                WHEN course LIKE '% (GER)' THEN 'Germany'
                WHEN course LIKE '% (USA)' THEN 'United States'
                WHEN course LIKE '% (CAN)' THEN 'Canada'
                WHEN course LIKE '% (UAE)' THEN 'United Arab Emirates'
                WHEN course LIKE '% (SAF)' THEN 'South Africa'
                WHEN course LIKE '% (NZ)' THEN 'New Zealand'
                WHEN course LIKE '% (JPN)' THEN 'Japan'
                WHEN course LIKE '% (BEL)' THEN 'Belgium'
                WHEN course LIKE '% (ITY)' THEN 'Italy'
                WHEN course LIKE '% (BRZ)' THEN 'Brazil'
                WHEN course LIKE '% (ARG)' THEN 'Argentina'
                WHEN course LIKE '% (CHI)' THEN 'Chile'
                WHEN course LIKE '% (JER)' THEN 'Jersey'
                WHEN course LIKE '% (GB)' THEN 'Great Britain'
                WHEN instr(course, ' (') = 0 THEN 'Great Britain or unmatched domestic'
                ELSE 'Other explicit suffix'
            END AS jurisdiction_group,
            CAST(substr(wgt, 1, instr(wgt, '-') - 1) AS INTEGER) AS stones,
            CAST(substr(wgt, instr(wgt, '-') + 1) AS INTEGER) AS pounds
        FROM data
        WHERE {DATA_ROW_PREDICATE}
    )
    SELECT
        jurisdiction_group,
        COUNT(*) AS runner_records,
        COUNT(DISTINCT course) AS distinct_courses,
        COUNT(DISTINCT raw_wgt) AS distinct_raw_values,
        MIN((stones * 14) + pounds) AS minimum_total_pounds,
        MAX((stones * 14) + pounds) AS maximum_total_pounds,
        SUM(
            CASE
                WHEN raw_wgt GLOB '[0-9]*-[0-9]*'
                 AND pounds BETWEEN 0 AND 13
                THEN 1
                ELSE 0
            END
        ) AS structurally_parseable_rows,
        COUNT(*) - SUM(
            CASE
                WHEN raw_wgt GLOB '[0-9]*-[0-9]*'
                 AND pounds BETWEEN 0 AND 13
                THEN 1
                ELSE 0
            END
        ) AS structurally_unresolved_rows
    FROM classified
    GROUP BY jurisdiction_group
    ORDER BY runner_records DESC, jurisdiction_group
    """,
    connection,
)

jurisdiction_weight_profile

,jurisdiction_group,runner_records,distinct_courses,distinct_raw_values,minimum_total_pounds,maximum_total_pounds,structurally_parseable_rows,structurally_unresolved_rows
0,Great Britain or unmatched domestic,749748,189,74,105,178,749748,0
1,Ireland,338817,27,74,96,179,338817,0
2,Other explicit suffix,293014,40,60,105,164,293014,0
3,France,214637,73,63,105,170,214637,0
4,Hong Kong,83068,2,33,103,135,83068,0
5,United States,48182,56,57,105,165,48182,0
6,Australia,43198,51,31,97,138,43198,0
7,United Arab Emirates,25178,5,27,110,137,25178,0
8,Japan,21202,21,24,106,139,21202,0
9,Germany,6173,17,26,110,140,6173,0


In [14]:
# Retrieve distinct course names from SQLite, then extract only a final
# parenthesised suffix in Python. This avoids relying on SQLite functions
# that are not available in the standard library build.

import re

distinct_courses = pd.read_sql_query(
    f"""
    SELECT DISTINCT course
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    ORDER BY course
    """,
    connection,
)

def extract_final_course_suffix(course: str) -> str:
    """Return a final parenthesised course suffix, or an explicit fallback."""
    match = re.search(r"\s+\(([^()]*)\)$", course)
    return match.group(1) if match else "[no suffix]"

course_suffix_profile = distinct_courses.copy()
course_suffix_profile["course_suffix"] = (
    course_suffix_profile["course"]
    .astype(str)
    .map(extract_final_course_suffix)
)

suffix_summary = (
    course_suffix_profile
    .groupby("course_suffix", as_index=False)
    .agg(distinct_courses=("course", "nunique"))
    .sort_values(
        ["distinct_courses", "course_suffix"],
        ascending=[False, True],
    )
)

display(suffix_summary)

print("\nCourses by extracted suffix:")
display(
    course_suffix_profile.sort_values(
        ["course_suffix", "course"],
        kind="stable",
    )
)

,course_suffix,distinct_courses
39,[no suffix],189
11,FR,73
38,USA,56
1,AUS,51
16,IRE,27
19,JPN,21
12,GER,17
24,NZ,14
29,SAF,9
2,AW,7



Courses by extracted suffix:


,course,course_suffix
246,La Plata (ARG),ARG
369,Palermo (ARG),ARG
420,San Isidro (ARG),ARG
5,Albury (AUS),AUS
6,Alice springs (AUS),AUS
...,...,...
517,Windsor,[no suffix]
521,Woodbine,[no suffix]
523,Worcester,[no suffix]
525,Yarmouth,[no suffix]


In [15]:
# Derive source-implied pounds and kilograms for every distinct raw weight.
#
# Kilograms are calculated only as the literal SI conversion of the stored
# stones-and-pounds expression:
#
#     1 pound = 0.45359237 kilograms
#
# This does not recover or verify an original official metric carried weight.

POUND_TO_KILOGRAM = 0.45359237

weight_conversion_profile = wgt_components[
    [
        "raw_wgt",
        "left_component",
        "right_component",
        "candidate_total_pounds",
        "runner_records",
    ]
].copy()

weight_conversion_profile = weight_conversion_profile.rename(
    columns={
        "left_component": "parsed_stones",
        "right_component": "parsed_pounds",
        "candidate_total_pounds": "source_implied_total_pounds",
    }
)

weight_conversion_profile["source_implied_kilograms"] = (
    weight_conversion_profile["source_implied_total_pounds"]
    * POUND_TO_KILOGRAM
)

# Show how closely each converted value lies to a whole or half kilogram.
# This helps demonstrate why reconversion can create apparent precision that
# may not have existed in a metric jurisdiction's original declaration.
weight_conversion_profile["nearest_whole_kg"] = (
    weight_conversion_profile["source_implied_kilograms"].round(0)
)

weight_conversion_profile["difference_from_whole_kg"] = (
    weight_conversion_profile["source_implied_kilograms"]
    - weight_conversion_profile["nearest_whole_kg"]
)

weight_conversion_profile["nearest_half_kg"] = (
    weight_conversion_profile["source_implied_kilograms"] * 2
).round(0) / 2

weight_conversion_profile["difference_from_half_kg"] = (
    weight_conversion_profile["source_implied_kilograms"]
    - weight_conversion_profile["nearest_half_kg"]
)

display(
    weight_conversion_profile.sort_values(
        "source_implied_total_pounds"
    )
)

display(
    pd.DataFrame(
        {
            "measure": [
                "minimum source-implied kilograms",
                "maximum source-implied kilograms",
                "values exactly equal to a whole kilogram",
                "values exactly equal to a half kilogram",
                "largest absolute difference from nearest whole kilogram",
                "largest absolute difference from nearest half kilogram",
            ],
            "value": [
                weight_conversion_profile[
                    "source_implied_kilograms"
                ].min(),
                weight_conversion_profile[
                    "source_implied_kilograms"
                ].max(),
                int(
                    weight_conversion_profile[
                        "difference_from_whole_kg"
                    ].abs().lt(1e-12).sum()
                ),
                int(
                    weight_conversion_profile[
                        "difference_from_half_kg"
                    ].abs().lt(1e-12).sum()
                ),
                weight_conversion_profile[
                    "difference_from_whole_kg"
                ].abs().max(),
                weight_conversion_profile[
                    "difference_from_half_kg"
                ].abs().max(),
            ],
        }
    )
)

,raw_wgt,parsed_stones,parsed_pounds,source_implied_total_pounds,runner_records,source_implied_kilograms,nearest_whole_kg,difference_from_whole_kg,nearest_half_kg,difference_from_half_kg
77,6-12,6,12,96,1,43.544868,44.0,-0.455132,43.5,0.044868
78,6-13,6,13,97,1,43.998460,44.0,-0.001540,44.0,-0.001540
73,7-5,7,5,103,9,46.720014,47.0,-0.279986,46.5,0.220014
72,7-6,7,6,104,13,47.173606,47.0,0.173606,47.0,0.173606
68,7-7,7,7,105,115,47.627199,48.0,-0.372801,47.5,0.127199
...,...,...,...,...,...,...,...,...,...,...
70,12-7,12,7,175,73,79.378665,79.0,0.378665,79.5,-0.121335
71,12-8,12,8,176,16,79.832257,80.0,-0.167743,80.0,-0.167743
74,12-9,12,9,177,3,80.285849,80.0,0.285849,80.5,-0.214151
75,12-10,12,10,178,1,80.739442,81.0,-0.260558,80.5,0.239442


,measure,value
0,minimum source-implied kilograms,43.544868
1,maximum source-implied kilograms,81.193034
2,values exactly equal to a whole kilogram,0.000000
3,values exactly equal to a half kilogram,0.000000
4,largest absolute difference from nearest whole...,0.497068
5,largest absolute difference from nearest half ...,0.249926


### Source-implied kilogram conversion

**Observation**

The 79 observed stones-and-pounds values correspond to source-implied weights from approximately `43.545 kg` to `81.193 kg`.

None of the literal pound-to-kilogram conversions lands exactly on a whole or half kilogram. The maximum difference from the nearest:

- whole kilogram is approximately `0.497 kg`;
- half kilogram is approximately `0.250 kg`.

**Interpretation**

This is expected when whole pounds are converted using the exact factor:

```text
1 lb = 0.45359237 kg

In [16]:
# Define a conservative carried-weight parser.
#
# Current accepted notation:
#     <stones>-<pounds>
#
# Validation rules:
# - the raw value must be a Python string;
# - it must use canonical integer-hyphen-integer notation;
# - neither component may contain leading zeros;
# - the pounds component must be between 0 and 13;
# - unfamiliar or malformed future values remain unresolved;
# - kilograms are labelled as source-implied conversions only.

from dataclasses import asdict, dataclass
from typing import Optional


POUND_TO_KILOGRAM = 0.45359237
CARRIED_WEIGHT_PATTERN = re.compile(r"^(0|[1-9]\d*)-(0|[1-9]\d*)$")


@dataclass(frozen=True)
class CarriedWeightParse:
    raw_wgt: object
    notation_family: str
    parsed_stones: Optional[int]
    parsed_pounds: Optional[int]
    source_implied_total_pounds: Optional[int]
    source_implied_kilograms: Optional[float]
    parse_status: str
    ambiguity_flag: bool
    anomaly_flags: tuple[str, ...]
    official_weight_verified: bool


def parse_carried_weight(raw_wgt: object) -> CarriedWeightParse:
    """Parse only canonical stones-and-pounds source notation."""

    if raw_wgt is None:
        return CarriedWeightParse(
            raw_wgt=raw_wgt,
            notation_family="missing",
            parsed_stones=None,
            parsed_pounds=None,
            source_implied_total_pounds=None,
            source_implied_kilograms=None,
            parse_status="unresolved_missing",
            ambiguity_flag=False,
            anomaly_flags=("missing_value",),
            official_weight_verified=False,
        )

    if not isinstance(raw_wgt, str):
        return CarriedWeightParse(
            raw_wgt=raw_wgt,
            notation_family="non_text",
            parsed_stones=None,
            parsed_pounds=None,
            source_implied_total_pounds=None,
            source_implied_kilograms=None,
            parse_status="unresolved_non_text",
            ambiguity_flag=True,
            anomaly_flags=("unexpected_storage_type",),
            official_weight_verified=False,
        )

    match = CARRIED_WEIGHT_PATTERN.fullmatch(raw_wgt)

    if match is None:
        return CarriedWeightParse(
            raw_wgt=raw_wgt,
            notation_family="unrecognised_text",
            parsed_stones=None,
            parsed_pounds=None,
            source_implied_total_pounds=None,
            source_implied_kilograms=None,
            parse_status="unresolved_unrecognised_notation",
            ambiguity_flag=True,
            anomaly_flags=("unrecognised_notation",),
            official_weight_verified=False,
        )

    stones = int(match.group(1))
    pounds = int(match.group(2))

    if pounds > 13:
        return CarriedWeightParse(
            raw_wgt=raw_wgt,
            notation_family="integer_hyphen_integer",
            parsed_stones=stones,
            parsed_pounds=pounds,
            source_implied_total_pounds=None,
            source_implied_kilograms=None,
            parse_status="unresolved_invalid_pounds_component",
            ambiguity_flag=False,
            anomaly_flags=("pounds_component_outside_0_to_13",),
            official_weight_verified=False,
        )

    total_pounds = (stones * 14) + pounds

    return CarriedWeightParse(
        raw_wgt=raw_wgt,
        notation_family="stones_and_pounds",
        parsed_stones=stones,
        parsed_pounds=pounds,
        source_implied_total_pounds=total_pounds,
        source_implied_kilograms=total_pounds * POUND_TO_KILOGRAM,
        parse_status="parsed",
        ambiguity_flag=False,
        anomaly_flags=(),
        official_weight_verified=False,
    )


# Apply the parser to all 79 distinct current source values.
current_weight_parse_results = pd.DataFrame(
    [
        asdict(parse_carried_weight(raw_wgt))
        for raw_wgt in wgt_value_profile["raw_wgt"]
    ]
)

display(
    current_weight_parse_results.groupby(
        ["notation_family", "parse_status"],
        as_index=False,
        dropna=False,
    ).size()
)

display(
    current_weight_parse_results.loc[
        current_weight_parse_results["parse_status"] != "parsed"
    ]
)

,notation_family,parse_status,size
0,stones_and_pounds,parsed,79


,raw_wgt,notation_family,parsed_stones,parsed_pounds,source_implied_total_pounds,source_implied_kilograms,parse_status,ambiguity_flag,anomaly_flags,official_weight_verified


In [17]:
# Exercise the parser against representative unseen future values.
# These cases verify that unsupported notation fails explicitly rather than
# being silently guessed or coerced.

future_value_tests = [
    # Valid canonical notation, including currently unobserved values.
    "7-0",
    "13-0",

    # Invalid pounds components.
    "9-14",
    "10-20",

    # Non-canonical variants.
    "09-0",
    "9-00",
    " 9-0",
    "9-0 ",
    "9 - 0",
    "9/0",

    # Possible pounds-only, metric, decimal, or fractional notation.
    "126",
    "126lb",
    "57kg",
    "57.0",
    "9-0½",
    "9-0.5",

    # Missing, blank, and unexpected storage types.
    "",
    None,
    126,
    57.0,
]

future_weight_parse_results = pd.DataFrame(
    [
        asdict(parse_carried_weight(raw_wgt))
        for raw_wgt in future_value_tests
    ]
)

display(
    future_weight_parse_results[
        [
            "raw_wgt",
            "notation_family",
            "parsed_stones",
            "parsed_pounds",
            "source_implied_total_pounds",
            "parse_status",
            "ambiguity_flag",
            "anomaly_flags",
        ]
    ]
)

# Failure-detecting assertions: only canonical stones-and-pounds values with
# a pounds component from 0 through 13 may parse successfully.
expected_parsed_values = {"7-0", "13-0"}

actual_parsed_values = set(
    future_weight_parse_results.loc[
        future_weight_parse_results["parse_status"] == "parsed",
        "raw_wgt",
    ]
)

assert actual_parsed_values == expected_parsed_values
assert (
    future_weight_parse_results.loc[
        ~future_weight_parse_results["raw_wgt"].isin(expected_parsed_values),
        "source_implied_total_pounds",
    ]
    .isna()
    .all()
)

print("Future-value failure-detection tests passed.")

,raw_wgt,notation_family,parsed_stones,parsed_pounds,source_implied_total_pounds,parse_status,ambiguity_flag,anomaly_flags
0,7-0,stones_and_pounds,7.0,0.0,98.0,parsed,False,()
1,13-0,stones_and_pounds,13.0,0.0,182.0,parsed,False,()
2,9-14,integer_hyphen_integer,9.0,14.0,NaN,unresolved_invalid_pounds_component,False,"(pounds_component_outside_0_to_13,)"
3,10-20,integer_hyphen_integer,10.0,20.0,NaN,unresolved_invalid_pounds_component,False,"(pounds_component_outside_0_to_13,)"
4,09-0,unrecognised_text,NaN,NaN,NaN,unresolved_unrecognised_notation,True,"(unrecognised_notation,)"
5,9-00,unrecognised_text,NaN,NaN,NaN,unresolved_unrecognised_notation,True,"(unrecognised_notation,)"
6,9-0,unrecognised_text,NaN,NaN,NaN,unresolved_unrecognised_notation,True,"(unrecognised_notation,)"
7,9-0,unrecognised_text,NaN,NaN,NaN,unresolved_unrecognised_notation,True,"(unrecognised_notation,)"
8,9 - 0,unrecognised_text,NaN,NaN,NaN,unresolved_unrecognised_notation,True,"(unrecognised_notation,)"
9,9/0,unrecognised_text,NaN,NaN,NaN,unresolved_unrecognised_notation,True,"(unrecognised_notation,)"


Future-value failure-detection tests passed.


In [18]:
# Validate complete source coverage at runner-record grain.
#
# This independently checks:
# - every current raw value matches canonical stones-and-pounds notation;
# - every pounds component lies between 0 and 13;
# - the Python parser succeeds for every distinct source value;
# - Python-derived total pounds agree with an independent SQLite calculation;
# - weighted record counts reconcile to the full data-like row count.

# Parse each distinct source value once, retaining its runner-record frequency.
distinct_weight_validation = wgt_value_profile[
    ["raw_wgt", "runner_records"]
].copy()

parsed_distinct_values = pd.DataFrame(
    [
        asdict(parse_carried_weight(raw_wgt))
        for raw_wgt in distinct_weight_validation["raw_wgt"]
    ]
)

distinct_weight_validation = distinct_weight_validation.merge(
    parsed_distinct_values,
    on="raw_wgt",
    how="left",
    validate="one_to_one",
)

# Calculate the same total independently in SQLite for each distinct value.
sql_weight_totals = pd.read_sql_query(
    f"""
    SELECT
        wgt AS raw_wgt,
        CAST(substr(wgt, 1, instr(wgt, '-') - 1) AS INTEGER) AS sql_stones,
        CAST(substr(wgt, instr(wgt, '-') + 1) AS INTEGER) AS sql_pounds,
        (
            CAST(substr(wgt, 1, instr(wgt, '-') - 1) AS INTEGER) * 14
            + CAST(substr(wgt, instr(wgt, '-') + 1) AS INTEGER)
        ) AS sql_total_pounds
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY wgt
    ORDER BY wgt
    """,
    connection,
)

distinct_weight_validation = distinct_weight_validation.merge(
    sql_weight_totals,
    on="raw_wgt",
    how="left",
    validate="one_to_one",
)

distinct_weight_validation["component_match"] = (
    distinct_weight_validation["parsed_stones"]
    == distinct_weight_validation["sql_stones"]
) & (
    distinct_weight_validation["parsed_pounds"]
    == distinct_weight_validation["sql_pounds"]
)

distinct_weight_validation["total_pounds_match"] = (
    distinct_weight_validation["source_implied_total_pounds"]
    == distinct_weight_validation["sql_total_pounds"]
)

validation_summary = pd.DataFrame(
    {
        "measure": [
            "data-like runner records",
            "runner records represented by distinct-value profile",
            "distinct raw values",
            "distinct values parsed",
            "runner records parsed",
            "distinct unresolved values",
            "runner records unresolved",
            "component mismatches against SQL",
            "total-pound mismatches against SQL",
        ],
        "value": [
            1_851_285,
            int(distinct_weight_validation["runner_records"].sum()),
            len(distinct_weight_validation),
            int(
                (
                    distinct_weight_validation["parse_status"] == "parsed"
                ).sum()
            ),
            int(
                distinct_weight_validation.loc[
                    distinct_weight_validation["parse_status"] == "parsed",
                    "runner_records",
                ].sum()
            ),
            int(
                (
                    distinct_weight_validation["parse_status"] != "parsed"
                ).sum()
            ),
            int(
                distinct_weight_validation.loc[
                    distinct_weight_validation["parse_status"] != "parsed",
                    "runner_records",
                ].sum()
            ),
            int((~distinct_weight_validation["component_match"]).sum()),
            int((~distinct_weight_validation["total_pounds_match"]).sum()),
        ],
    }
)

display(validation_summary)

# Failure-detecting reconciliation assertions.
assert distinct_weight_validation["runner_records"].sum() == 1_851_285
assert (
    distinct_weight_validation["parse_status"] == "parsed"
).all()
assert distinct_weight_validation["component_match"].all()
assert distinct_weight_validation["total_pounds_match"].all()

print("Complete source carried-weight validation passed.")

,measure,value
0,data-like runner records,1851285
1,runner records represented by distinct-value p...,1851285
2,distinct raw values,79
3,distinct values parsed,79
4,runner records parsed,1851285
5,distinct unresolved values,0
6,runner records unresolved,0
7,component mismatches against SQL,0
8,total-pound mismatches against SQL,0


Complete source carried-weight validation passed.


## Complete-source validation result

### Observation

The conservative carried-weight parser reconciles completely against the current source:

- **Data-like runner records:** 1,851,285
- **Distinct raw `wgt` values:** 79
- **Distinct values parsed:** 79
- **Runner records parsed:** 1,851,285
- **Unresolved current values:** 0
- **Component mismatches against independent SQL parsing:** 0
- **Total-pound mismatches against independent SQL calculation:** 0

All current values use canonical integer-hyphen-integer notation. The left component ranges from 6 to 12 and the right component ranges from 0 to 13.

### Interpretation

The current `wgt` field can be parsed reproducibly as stones and pounds:

\[
\text{source-implied total pounds}
=
(\text{stones} \times 14) + \text{pounds}
\]

The evidence supporting this interpretation is:

- every current record uses the same notation structure;
- the second component never exceeds 13;
- all raw values are canonical and map one-to-one to total pounds;
- resulting ranges are coherent with Flat, Hurdle, Chase and NH Flat contexts;
- unusual extreme values remain internally coherent within their races;
- clearly metric international jurisdictions still use the same imperial notation.

### Provenance limitation

The raw source does not preserve native metric notation for jurisdictions that ordinarily publish carried weight in kilograms.

A literal conversion may therefore be derived as:

\[
\text{source-implied kilograms}
=
\text{source-implied total pounds} \times 0.45359237
\]

However, this value is only the SI equivalent of the stored source expression. It must not be presented as a recovered or independently verified official metric carried weight.

For example, a stored value of `9-0` implies exactly 126 pounds and approximately 57.1526 kilograms, but the original official declaration may have been a rounded metric value such as 57 kilograms.

### Parsing decision

Preserve separately:

- exact raw `wgt`;
- notation family;
- parsed stones;
- parsed pounds;
- source-implied total pounds;
- source-implied kilograms;
- parse status;
- ambiguity and anomaly flags;
- `official_weight_verified = False`.

Parse only canonical stones-and-pounds notation with a pounds component from 0 through 13.

Do not silently trim, normalise or reinterpret:

- whitespace variants;
- leading-zero variants;
- pounds-only values;
- metric notation;
- decimal notation;
- fractional notation;
- non-text values;
- values whose pounds component exceeds 13.

Unfamiliar future values must remain unresolved for review rather than being guessed.

## Notebook conclusion

### Defensible parsing rule

The source `wgt` field is consistently represented as canonical stones-and-pounds text:

    <stones>-<pounds>

A value is deterministically parseable when:

- it is stored as text;
- it contains exactly two unsigned integer components separated by one hyphen;
- neither component uses leading zeros;
- the pounds component is between `0` and `13`.

Derived values are:

    source_implied_total_pounds = (parsed_stones × 14) + parsed_pounds
    source_implied_kilograms = source_implied_total_pounds × 0.45359237

### Current-source result

Across all 1,851,285 data-like runner records:

- all values are SQLite text;
- there are no SQL `NULL` or blank values;
- there are 79 distinct raw values;
- all 79 values use canonical stones-and-pounds notation;
- all 1,851,285 runner records parse successfully;
- no current values remain unresolved;
- independent Python and SQLite calculations agree for every observed value.

The observed range is:

    6-12 to 12-11
    96 to 179 source-implied pounds
    43.544868 to 81.193034 source-implied kilograms

### Known exceptions and contextual anomalies

No malformed, fractional, pounds-only or metric notation occurs in the current source.

Some unusual weights and race-level spreads occur, but inspection shows that the weight strings themselves remain structurally valid and internally coherent. Contextual anomalies in fields such as race type must therefore remain separate from weight parse validity.

### International provenance limitation

The source uses stones-and-pounds notation even for jurisdictions that ordinarily publish carried weight in kilograms.

Consequently:

- raw `wgt` is exact source evidence;
- total pounds are the exact interpretation of the stored expression;
- kilograms are a deterministic SI conversion of that expression;
- converted kilograms are suitable for consistent analysis;
- converted kilograms must not be described as independently verified official metric declarations;
- exact original metric values may have been rounded before storage and cannot necessarily be reconstructed.

### Preservation mechanism

Preserve separately:

- exact raw `wgt`;
- detected notation family;
- parsed stones;
- parsed pounds;
- source-implied total pounds;
- source-implied kilograms;
- parse status;
- ambiguity flag;
- anomaly flags;
- `official_weight_verified = False`.

### Future-value rule

Unseen future values must remain unresolved when they contain:

- non-text storage;
- missing or blank values;
- surrounding or internal whitespace;
- leading zeros;
- invalid pounds components;
- pounds-only notation;
- metric unit text;
- decimal or fractional notation;
- any other unfamiliar structure.

The parser must not silently trim, normalise or guess.

### Stopping decision

The notebook has established:

1. a reproducible parsing rule covering every current value;
2. known contextual exceptions without confusing them with parse failures;
3. explicit preservation of raw evidence and provenance;
4. conservative handling of unseen future notation;
5. failure-detecting validation against malformed examples and the complete source.

The bounded question is answered. No further carried-weight investigation is required before clean-kernel validation.